# Episode 7 — Valuing a live swap

Companion notebook for the video. A swap traded in July 2025 is valued on 22 September 2026: two payments are already made, the current 6-month BBSW rate is already set, and the rest is forecast from today's curves. We value it in QuantLib, split the value into two pieces we can check by hand, and separate accrued interest from the clean value. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** Today's quotes (`quotes_illustrative.csv`, as in Episode 6), the trade's fixed rate and the 6-month BBSW fixing of 15 July 2026 are made up for teaching. They are **not market data** (BBSW fixings are licensed ASX data). Educational material only, not investment advice.

1. Setup · 2. The trade · 3. Today's curves · 4. The past fixing · 5. Value today · 6. Remaining cash flows · 7. Two pieces · 8. Accrued interest · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
trade_date = "2025-07-14"
tenor = "5Y"
fixed_rate_pct = 3.95              # illustrative
current_fixing_date = "2026-07-15"
current_fixing_pct = 4.20          # illustrative 6M BBSW fixing

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()
cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)

out = {"meta": {
    "episode": 7, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

The curve-building code shared by Episodes 6 to 9:

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

## 2. The trade

Receive fixed on a 5-year AUD swap against 6-month BBSW, semi-annual on both legs, traded on 14 July 2025 and starting the next business day.

In [ ]:
trade = ql.DateParser.parseISO(trade_date)
start = cal.advance(trade, 1, ql.Days)
K = fixed_rate_pct / 100

curves, H, quote_handles, helpers = build_curves(pd.read_csv(quotes_file))
swap = vanilla_swap(tenor, K, 6, H["BBSW6M"], H["AONIA"], notional, receive=True, start=start)
sched = [c.accrualStartDate() for c in map(ql.as_coupon, swap.fixedLeg())] + [ql.as_coupon(swap.fixedLeg()[len(swap.fixedLeg()) - 1]).accrualEndDate()]
paid = [d for d in sched[1:] if d <= today]
out["trade"] = {"trade_date": iso(trade), "start": iso(start), "maturity": iso(sched[-1]), "notional": notional,
                "fixed_rate_pct": fixed_rate_pct, "side": "receive fixed", "n_periods": len(sched) - 1,
                "n_paid": len(paid), "last_paid": iso(paid[-1]),
                "years_left": dc.yearFraction(today, sched[-1])}
pd.Series(out["trade"])

## 3. Today's curves

Built from today's illustrative quotes exactly as in Episode 6: AONIA discounts; 6-month BBSW forecasts.

## 4. The past fixing

The current period started on 15 July 2026, and its 6-month BBSW rate was set that day. It is history now, so it can't come from today's curve. QuantLib refuses to price the swap until we tell it the fixing.

In [ ]:
try:
    swap.NPV()
    missing_error = None
except RuntimeError as e:
    missing_error = str(e).splitlines()[0].split(": ", 1)[-1]   # drop QuantLib's "2nd leg: " prefix
print("without the fixing:", missing_error)

fixing_date = ql.DateParser.parseISO(current_fixing_date)
bbsw(6).addFixing(fixing_date, current_fixing_pct / 100)   # fixings are stored by index name
out["fixing"] = {"date": current_fixing_date, "rate_pct": current_fixing_pct, "missing_error": missing_error}

## 5. Value today

In [ ]:
value = {"npv": swap.NPV(), "fixed_leg": swap.fixedLegNPV(), "float_leg": swap.floatingLegNPV()}
out["value"] = value
pd.Series(value)

## 6. Remaining cash flows

Periods whose payment date is after today. The first one's floating rate is the fixing; the rest are 6-month BBSW forwards.

In [ ]:
rows = []
for f, fl in zip(swap.fixedLeg(), swap.floatingLeg()):
    if fl.date() <= today:
        continue
    c = ql.as_floating_rate_coupon(fl)
    df = curves["AONIA"].discount(fl.date())
    rows.append({"accrual_start": iso(c.accrualStartDate()), "accrual_end": iso(c.accrualEndDate()),
                 "payment_date": iso(fl.date()), "days": c.accrualDays(),
                 "float_rate_pct": c.indexFixing() * 100, "known": c.fixingDate() <= today,
                 "fixed_amount": f.amount(), "float_amount": fl.amount(), "df": df,
                 "pv_net": (f.amount() - fl.amount()) * df})
cf = pd.DataFrame(rows)
out["cashflows"] = {"rows": rows, "n": len(rows), "sum_pv_net": float(cf.pv_net.sum())}
cf

## 7. Two pieces

Split the swap at the next reset date $t_1$:

* **The current period**: both rates are known, so it is one net payment, $N (K - R_0)\,\tau_0$, discounted from its payment date.
* **The rest**: a forward-starting swap from $t_1$ to maturity. Its value is $N (K - S_\text{fwd})\,A_\text{fwd}$, where $S_\text{fwd}$ is today's par rate for those dates and $A_\text{fwd}$ is their annuity on AONIA.

In [ ]:
c0 = ql.as_floating_rate_coupon(next(fl for fl in swap.floatingLeg() if fl.date() > today))
tau0 = c0.accrualPeriod()
current = notional * (K - current_fixing_pct / 100) * tau0 * curves["AONIA"].discount(c0.date())

rest = [d for d in sched if d >= c0.accrualEndDate()]
fwd_sched = ql.Schedule(rest, cal, ql.ModifiedFollowing)
fwd = ql.VanillaSwap(ql.VanillaSwap.Receiver, notional, fwd_sched, K, dc, fwd_sched, bbsw(6, H["BBSW6M"]), 0.0, dc)
fwd.setPricingEngine(ql.DiscountingSwapEngine(H["AONIA"]))
S_fwd = fwd.fairRate()
A_fwd = abs(fwd.fixedLegBPS()) / (notional * 1e-4)
rest_value = notional * (K - S_fwd) * A_fwd
pieces = {"current_days": c0.accrualDays(), "current_pay": iso(c0.date()), "current_pv": current,
          "forward_start": iso(rest[0]), "s_fwd_pct": S_fwd * 100, "a_fwd": A_fwd, "rest_pv": rest_value,
          "total_hand": current + rest_value, "total_ql": value["npv"]}
pieces["abs_diff"] = abs(pieces["total_hand"] - pieces["total_ql"])
out["pieces"] = pieces
pd.Series(pieces)

## 8. Accrued interest

Since 15 July both legs have been accruing. The value above includes that accrued interest (a *dirty* value). Taking it out gives the *clean* value, which moves only with market rates, not with the calendar.

In [ ]:
acc_days = today - c0.accrualStartDate()
acc_fixed_hand = notional * K * acc_days / 365
acc_float_hand = notional * current_fixing_pct / 100 * acc_days / 365
acc_fixed_ql = ql.CashFlows.accruedAmount(swap.fixedLeg(), False, today)
acc_float_ql = ql.CashFlows.accruedAmount(swap.floatingLeg(), False, today)
accrued = {"days": acc_days, "fixed_hand": acc_fixed_hand, "float_hand": acc_float_hand,
           "fixed_ql": acc_fixed_ql, "float_ql": acc_float_ql,
           "net": acc_fixed_hand - acc_float_hand}
accrued["dirty"] = value["npv"]
accrued["clean"] = value["npv"] - accrued["net"]
out["accrued"] = accrued
pd.Series(accrued)

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")